# 21b - Health check before documentation (Part B: the phone's photos and blur)

Part A (notebook 21) used saved scores. Those scores were computed on photos shrunk to 256 px
on the shorter side for training speed, but the app sends something else: the phone shrinks
the photo to at most 1024 px on the longer side and saves it as JPEG quality 80, and the
server saves it again at quality 85. Blur and compression cannot be studied from saved scores,
so here the three models look at the photos again. **Needs a GPU runtime (T4 is enough),
about 20 minutes.**

**Photos:** the field photos (FieldPlant and the Tanzanian potato set) of the validation and
test splits, read in full size from the raw folders. Validation photos are used for every
choice below; test photos are reported only. (The selection set was kept only at 256 px, and
PlantVillage photos are 256 px originals, so neither can show the effect of size.)

**The app's rule throughout** (notebook 21 A3, now deployed): confidence within the farmer's
crop (temperature 1.7002), referred below 0.9557 or when the crop holds under half of the
11-class probability (temperature 2.4381). The farmer's crop is given.

**Rules, written on 31 Oct 2026 before this notebook was first run:**

- **B1 Does the phone's photo change the result?** Compare the phone's version (1024 px,
  quality 80, then the server's quality 85) with the version the saved scores came from
  (256 px, quality 95). If, on validation field photos, the phone's accuracy is more than 1
  point lower **and** the paired bootstrap range of the difference lies wholly below zero, the
  app's setting changes to the smallest setting in the grid below (largest side first, then
  quality) whose accuracy is within 1 point of the evaluation version; otherwise it stays.
  The whole grid (longest side 1600, 1024, 512, 256 px; quality 90, 80, 60, 40, 20) is
  reported on test photos for the supervisor's compression question.
- **B2 Should blurred photos be stopped with "Retake: too blurred"?** Sharpness is the
  variance of the Laplacian of the grey image shrunk to 512 px on the longer side
  (src/blur_check.py; the backend would compute the same number). The cut-off is fixed now as
  the value that 99% of the phone versions of real validation field photos pass. Photos are
  then blurred (Gaussian, radius 1, 2, 3, 5 and 8 px at 1024 px) before the phone's JPEG step.
  **The check is switched on only if**, at the mildest blur whose validation accuracy is at
  least 10 points below the sharp photos, it stops at least half of those photos. Otherwise it
  would block farmers without catching the blur that matters, and it stays off (reported).

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings, photos and models

In [ ]:
import importlib
import io
import json
import shutil

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFilter
from torchvision import transforms
from tqdm.auto import tqdm

from src import blur_check, calibration as cal, metrics, training
for module in (blur_check, cal, metrics, training):
    importlib.reload(module)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Runtime > Change runtime type > T4 GPU"
BOOTSTRAP, SEED = 1000, config.RANDOM_SEED

health = json.loads((ML_DIR / "results" / "health_check" / "decision.json").read_text())
assert health["A3"]["adopted"], "notebook 21 A3 was not adopted; this notebook assumes the within-crop rule"
MEMBERS = health["deployed"]["model_runs"]
TEMPERATURE = health["deployed"]["temperature"]
CROP_TEMPERATURE = health["A3"]["temperature_within_crop"]
THRESHOLD = health["A3"]["threshold"]

SIDES = [1600, 1024, 512, 256]
QUALITIES = [90, 80, 60, 40, 20]
BLURS = [1, 2, 3, 5, 8]
APP_SIDE, APP_QUALITY, SERVER_QUALITY = 1024, 80, 85
OUT_DIR = config.MODELS_DIR / "health_check_photos"
REPO_RESULTS = ML_DIR / "results" / "health_check_photos"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_RESULTS.mkdir(parents=True, exist_ok=True)

CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])
FIELD_SOURCES = ["fieldplant", "tanzania"]

PHOTOS = {}
for name in ["val", "test"]:
    rows = pd.read_csv(config.SPLITS_DIR / f"{name}.csv")
    rows = rows[rows["source"].isin(FIELD_SOURCES) & rows["class"].isin(INDEX)].reset_index(drop=True)
    rows["file"] = [config.RAW_DATA_DIR / p for p in rows["path"]]
    missing = int((~rows["file"].map(Path.exists)).sum())
    assert missing == 0, f"{missing} {name} photos are missing from {config.RAW_DATA_DIR}"
    rows["label"] = rows["class"].map(INDEX)
    rows["crop"] = CLASS_CROPS[rows["label"].to_numpy()]
    PHOTOS[name] = rows
print({n: len(r) for n, r in PHOTOS.items()}, "field photos | T =", TEMPERATURE, "| crop T =", CROP_TEMPERATURE,
      "| threshold =", round(THRESHOLD, 4))

In [ ]:
class Member:
    """One exported model.pt with the preprocessing it was evaluated with (as the backend loads it)."""

    def __init__(self, path):
        saved = torch.load(path, map_location="cpu", weights_only=False)
        assert saved["classes"] == CLASSES, path
        self.network = training.build_model(len(CLASSES), saved["architecture"], pretrained=False)
        self.network.load_state_dict(saved["state_dict"])
        self.network = self.network.to(DEVICE).eval()
        size = saved["image_size"]
        self.transform = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                                             transforms.Normalize(saved["mean"], saved["std"])])


MODELS = [Member(config.MODELS_DIR / run / "model.pt") for run in MEMBERS]


@torch.no_grad()
def ensemble_scores(images):
    """Log of the members' averaged probabilities, exactly as the backend combines them."""
    probs = []
    for m in MODELS:
        batch = torch.stack([m.transform(im) for im in images]).to(DEVICE)
        probs.append(torch.softmax(m.network(batch).float(), dim=1))
    return torch.log(torch.stack(probs).mean(dim=0) + 1e-12).cpu().numpy()

## Every version of every photo

Each photo is read once and turned into all its versions, which the three models then score
together. Sharpness is measured on what the server would store.

In [ ]:
def jpeg(image, quality):
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    with Image.open(buffer) as again:
        return again.convert("RGB")


def longest(image, side):
    image = image.copy()
    image.thumbnail((side, side), Image.BICUBIC)
    return image


def evaluation_version(image):
    """How the saved scores were made (src/data/image_dataset.py): shorter side 256, quality 95."""
    scale = 256 / min(image.size)
    if scale < 1:
        image = image.resize((round(image.width * scale), round(image.height * scale)), Image.BICUBIC)
    return jpeg(image, 95)


def app_version(image, side=APP_SIDE, quality=APP_QUALITY, blur=0):
    """The phone (longest side, JPEG quality), an optional blur before it, then the server's quality 85."""
    image = longest(image, side)
    if blur:
        image = image.filter(ImageFilter.GaussianBlur(blur))
    return jpeg(jpeg(image, quality), SERVER_QUALITY)


VERSIONS = {"evaluation": evaluation_version}
VERSIONS.update({f"{s} px, q{q}": (lambda im, s=s, q=q: app_version(im, s, q)) for s in SIDES for q in QUALITIES})
VERSIONS.update({f"blur {r}": (lambda im, r=r: app_version(im, blur=r)) for r in BLURS})
APP = f"{APP_SIDE} px, q{APP_QUALITY}"

SCORES = {name: {v: np.zeros((len(rows), len(CLASSES)), np.float32) for v in VERSIONS} for name, rows in PHOTOS.items()}
SHARPNESS = {name: {v: np.zeros(len(rows)) for v in VERSIONS} for name, rows in PHOTOS.items()}
for name, rows in PHOTOS.items():
    for i, file in enumerate(tqdm(rows["file"], desc=name)):
        with Image.open(file) as original:
            original = original.convert("RGB")
            made = {v: make(original) for v, make in VERSIONS.items()}
        scores = ensemble_scores(list(made.values()))
        for k, v in enumerate(made):
            SCORES[name][v][i] = scores[k]
            SHARPNESS[name][v][i] = blur_check.sharpness(made[v])
np.savez_compressed(OUT_DIR / "scores.npz", **{f"{n}|{v}": s for n in SCORES for v, s in SCORES[n].items()})
np.savez_compressed(OUT_DIR / "sharpness.npz", **{f"{n}|{v}": s for n in SHARPNESS for v, s in SHARPNESS[n].items()})

In [ ]:
NEG = -1e4


def app_rule(scores, rows):
    """The deployed rule: (confidence within the crop, correct, answered)."""
    crops, labels = rows["crop"].to_numpy(), rows["label"].to_numpy()
    keep = CLASS_CROPS[None, :] == crops[:, None]
    share = (metrics.softmax(scores / TEMPERATURE) * keep).sum(axis=1)
    within = metrics.softmax(np.where(keep, scores, NEG) / CROP_TEMPERATURE)
    conf, ok = within.max(axis=1), within.argmax(axis=1) == labels
    return conf, ok, (share >= 0.5) & (conf >= THRESHOLD)


def summary(name, version, passed=None):
    conf, ok, answered = app_rule(SCORES[name][version], PHOTOS[name])
    row = {"photos": len(ok), "accuracy (crop given)": ok.mean(), "mean confidence": conf.mean(),
           "ECE": metrics.expected_calibration_error(conf, ok), "answered": answered.mean(),
           "accuracy of answered": ok[answered].mean() if answered.any() else np.nan,
           "wrong per 100": 100 * (answered & ~ok).mean()}
    if passed is not None:
        row["stopped by blur check"] = 1 - passed.mean()
        kept = answered & passed
        row["answered after blur check"] = kept.mean()
        row["wrong per 100 after blur check"] = 100 * (kept & ~ok).mean()
    return row


def accuracy_change(name, a, b):
    """Accuracy of b minus a on the same photos, with the paired bootstrap range."""
    ok_a, ok_b = app_rule(SCORES[name][a], PHOTOS[name])[1], app_rule(SCORES[name][b], PHOTOS[name])[1]
    low, high = cal.bootstrap(lambda x, y: y.mean() - x.mean(), ok_a, ok_b, n=BOOTSTRAP, seed=SEED)
    return ok_b.mean() - ok_a.mean(), low, high

## B1 - The phone's photo against the evaluation version (rule fixed above)

In [ ]:
change, low, high = accuracy_change("val", "evaluation", APP)
B1_WORSE = bool(change < -0.01 and high < 0)
order = [f"{s} px, q{q}" for s in SIDES for q in QUALITIES]   # largest first, then highest quality
if B1_WORSE:
    eval_acc = summary("val", "evaluation")["accuracy (crop given)"]
    within = [v for v in order if summary("val", v)["accuracy (crop given)"] >= eval_acc - 0.01]
    NEW_SETTING = within[-1] if within else order[0]
else:
    NEW_SETTING = APP
B1 = {"validation_accuracy_change_app_minus_evaluation": change, "range": [low, high], "app_worse": B1_WORSE,
      "app_setting": NEW_SETTING, "changed": NEW_SETTING != APP}
print(json.dumps(B1, indent=1, default=float))

b1 = pd.DataFrame([{"set": n, "version": v, **summary(n, v)} for n in ["val", "test"] for v in ["evaluation", APP]])
b1 = b1.set_index(["set", "version"])
display(b1.round(4))
grid = pd.DataFrame([{"longest side": s, "quality": q, **summary("test", f"{s} px, q{q}")} for s in SIDES for q in QUALITIES])
grid = grid.set_index(["longest side", "quality"])
print("Test field photos, every size and quality (reported only)")
display(grid.round(4))

## B2 - Blur and the sharpness check (rule fixed above)

In [ ]:
CUT_OFF = float(np.percentile(SHARPNESS["val"][APP], 100 * (1 - blur_check.PASS_RATE)))
sharp_acc = summary("val", APP)["accuracy (crop given)"]
harmful = [r for r in BLURS if summary("val", f"blur {r}")["accuracy (crop given)"] <= sharp_acc - 0.10]
if harmful:
    mildest = harmful[0]
    stopped = float((SHARPNESS["val"][f"blur {mildest}"] < CUT_OFF).mean())
    SWITCH_ON = stopped >= 0.5
else:
    mildest, stopped, SWITCH_ON = None, None, False
B2 = {"cut_off": CUT_OFF, "pass_rate_rule": blur_check.PASS_RATE, "mildest_harmful_blur": mildest,
      "stopped_at_mildest_harmful_blur": stopped, "switched_on": SWITCH_ON}
print(json.dumps(B2, indent=1, default=float))

b2_rows = []
for n in ["val", "test"]:
    for v in [APP] + [f"blur {r}" for r in BLURS]:
        b2_rows.append({"set": n, "version": "sharp (phone)" if v == APP else v,
                        **summary(n, v, passed=SHARPNESS[n][v] >= CUT_OFF)})
b2 = pd.DataFrame(b2_rows).set_index(["set", "version"])
display(b2.round(4))

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED = "#0b0b0b", "#52514e"
with Image.open(PHOTOS["test"]["file"][0]) as original:
    original = original.convert("RGB")
    shown = [("sharp (phone)", app_version(original))] + [(f"blur {r} px", app_version(original, blur=r)) for r in BLURS]
fig, axes = plt.subplots(1, len(shown), figsize=(3 * len(shown), 3.4))
for ax, (title, image) in zip(axes, shown):
    ax.imshow(image)
    ax.set_title(f"{title}\nsharpness {blur_check.sharpness(image):.0f}", fontsize=9, color=INK)
    ax.axis("off")
fig.suptitle(f"Blur levels (cut-off {CUT_OFF:.0f}: below it the photo would be refused)", color=INK, fontsize=10, x=0.01, ha="left")
plt.tight_layout()
plt.savefig(OUT_DIR / "blur_levels.png", dpi=120)
plt.show()

## Save the results

In [ ]:
b1.to_csv(OUT_DIR / "b1_app_vs_evaluation.csv")
grid.to_csv(OUT_DIR / "b1_size_and_quality.csv")
b2.to_csv(OUT_DIR / "b2_blur.csv")
blur_check.export(CUT_OFF, OUT_DIR / "blur_check.json",
                  extra={"switched_on": SWITCH_ON, "fitted_on": f"validation field photos ({len(PHOTOS['val'])}), phone version"})
training.save_json({"B1": B1, "B2": B2, "rules_written": "2026-10-31, before the first run",
                    "app_rule": {"temperature": TEMPERATURE, "crop_temperature": CROP_TEMPERATURE, "threshold": THRESHOLD}},
                   OUT_DIR / "decision.json")
print("B1:", f"phone setting changes to {NEW_SETTING}" if B1["changed"] else "the phone's setting stays (1024 px, quality 80)")
print("B2:", f"blur check ON, cut-off {CUT_OFF:.1f}" if SWITCH_ON else "blur check stays OFF (reported)")

## Copy the results into the repository, sync Drive, save to GitHub

Save the notebook first (Ctrl+S).

In [ ]:
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json", ".png"}:
        shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

from src import github_sync

# Drive can hide a folder made just before the remount, so make sure it is there.
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
shutil.copy(ML_DIR / "notebooks" / "21b_health_check_photos.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/health-check-b",
    message="Health check Part B: the phone's photos and blur",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))